In [1]:
#Importing Libraries

from pathlib import Path
import pandas as pd
import random
import json
from collections import Counter

In [2]:
# Confirming and setting project path

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "Data" / "Raw" #Contains datasets csv
print(BASE_DIR)
print(RAW_DIR)


D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Raw


In [3]:
#Confirm files and shape

diseases = pd.read_csv(RAW_DIR / "diseases.csv")
symptoms = pd.read_csv(RAW_DIR / "symptoms.csv")
abbreviations = pd.read_csv(RAW_DIR / "abbreviations.csv")
medications = pd.read_csv(RAW_DIR / "medications.csv")
procedures = pd.read_csv(RAW_DIR / "procedures.csv")
dosages = pd.read_csv(RAW_DIR / "dosages.csv")

print(diseases.shape)
print(symptoms.shape)
print(medications.shape)
print(procedures.shape)
print(dosages.shape)
print(abbreviations.shape)

(100, 1)
(50, 1)
(50, 1)
(50, 1)
(12, 1)
(30, 3)


In [4]:
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
#Final Dump to create templates + save as json and csv file

# =====================================================
# LOAD VOCABULARIES
# =====================================================

diseases = diseases["disease"].dropna().tolist()
symptoms = symptoms["symptom"].dropna().tolist()
abbreviations = pd.read_csv(RAW_DIR / "abbreviations.csv")
medications = medications["medication"].dropna().tolist()
procedures = procedures["procedure"].dropna().tolist()
dosages = dosages["dosage"].dropna().tolist()

# Verify

In [6]:
print(pd.isna(diseases).sum())

0


In [7]:
print(pd.isna(symptoms).sum())

0


In [8]:
print(pd.isna(abbreviations).sum())

abbreviation    0
meaning         0
entity_type     0
dtype: int64


In [9]:
print(pd.isna(medications).sum())

0


In [10]:
print(pd.isna(procedures).sum())

0


In [11]:
print(pd.isna(dosages).sum())

0


In [12]:
# =====================================================
# GET TEMPLATES
# =====================================================

templates = [

    "Patient presents with {SYMPTOM}. Diagnosed with {DISEASE}. Prescribed {MEDICATION} {DOSAGE}.",

    "Patient reports {SYMPTOM} and {SYMPTOM2}. Assessment indicates {DISEASE}. Started on {MEDICATION} {DOSAGE}.",

    "Chief complaint: {SYMPTOM}. Diagnosis: {DISEASE}. Treatment initiated with {MEDICATION} {DOSAGE}.",

    "Patient evaluated for {SYMPTOM}. Clinical findings suggest {DISEASE}. Recommended {MEDICATION} {DOSAGE}.",

    "Presented with worsening {SYMPTOM}. Confirmed diagnosis of {DISEASE}. Prescribed {MEDICATION} {DOSAGE}.",

    "Follow-up visit for {DISEASE}. Patient continues to experience {SYMPTOM}. Continue {MEDICATION} {DOSAGE}.",

    "Patient returns for review of {DISEASE}. Reports improvement in {SYMPTOM}. Maintain {MEDICATION} {DOSAGE}.",

    "Follow-up assessment completed. Persistent {SYMPTOM} noted. Medication adjusted to {MEDICATION} {DOSAGE}.",

    "Known history of {DISEASE}. Current complaint includes {SYMPTOM}. Continue treatment with {MEDICATION} {DOSAGE}.",

    "Assessment: {DISEASE}. Plan: Start {MEDICATION} {DOSAGE} for management of {SYMPTOM}.",

    "Assessment reveals {DISEASE}. Plan includes treatment with {MEDICATION} {DOSAGE}. Patient reports {SYMPTOM}.",

    "Clinical impression: {DISEASE}. Symptoms include {SYMPTOM}. Initiate {MEDICATION} {DOSAGE}.",

    "Primary diagnosis: {DISEASE}. Associated symptom: {SYMPTOM}. Plan to prescribe {MEDICATION} {DOSAGE}.",

    "Patient underwent {PROCEDURE}. Findings consistent with {DISEASE}.",

    "{PROCEDURE} performed due to {SYMPTOM}. Results indicate {DISEASE}.",

    "Diagnostic evaluation included {PROCEDURE}. Assessment confirmed {DISEASE}.",

    "Patient scheduled for {PROCEDURE}. Reason for procedure: {SYMPTOM}.",

    "Started on {MEDICATION} {DOSAGE} for treatment of {DISEASE}.",

    "Medication prescribed: {MEDICATION} {DOSAGE}. Indication: {DISEASE}.",

    "Patient advised to take {MEDICATION} {DOSAGE}. Diagnosis remains {DISEASE}.",
    
    "Patient has {DISEASE}. Advised to take {MEDICATION} {DOSAGE}.",
    
    "Known case of {DISEASE}. Advised to take {MEDICATION} {DOSAGE}.",
    
    "History of {DISEASE}. Advised to take {MEDICATION} {DOSAGE}.",
    
    "Patient suffers from {DISEASE}. Advised to take {MEDICATION} {DOSAGE}.",
    
    "Assessment suggests {DISEASE}. Advised to take {MEDICATION} {DOSAGE}."

]

In [13]:
import random

def random_case(text):
    """
    Randomly vary capitalization to reduce dataset bias.
    """

    r = random.random()

    if r < 0.33:
        return text.lower()

    elif r < 0.66:
        return text.title()

    return text


# Alternate
# choice = random.choice(["lower", "title", "original"])

# if choice == "lower":
#     return text.lower()
# elif choice == "title":
#     return text.title()
# else:
#     return text

In [14]:
# =====================================================
# HELPER FUNCTION
# =====================================================

def add_entity(text, value, label, entities):

    start = text.find(value)

    if start == -1:
        return

    end = start + len(value)

    entities.append({
    "start": start,
    "end": end,
    "label": label
    })
    

In [15]:
# =====================================================
# ABBREVIATION LOOKUP
# =====================================================

abbrev_dict = dict(
    zip(
        abbreviations["meaning"],
        abbreviations["abbreviation"]
    )
)

In [16]:
records = []

N_RECORDS = 5000

for record_id in range(1, N_RECORDS + 1):

    template = random.choice(templates)

    symptom1 = random.choice(symptoms)

    # IMPORTANT FIX
    symptom2 = None

    if "{SYMPTOM2}" in template:
        symptom1, symptom2 = random.sample(symptoms, 2)

    # =====================================================
    # DISEASE SELECTION
    # =====================================================

    disease = random.choice(diseases)

    if disease in abbrev_dict:
        disease = random.choice([
            disease,
            abbrev_dict[disease]
        ])

    disease = random_case(disease)

    symptom = random_case(random.choice(symptoms))
    medication = random_case(random.choice(medications))
    procedure = random_case(random.choice(procedures))
    dosage = random.choice(dosages)

    text = template

    replacements = {
        "{SYMPTOM}": symptom1,
        "{DISEASE}": disease,
        "{MEDICATION}": medication,
        "{DOSAGE}": dosage,
        "{PROCEDURE}": procedure
    }

    if symptom2:
        replacements["{SYMPTOM2}"] = symptom2

    for key, value in replacements.items():
        text = text.replace(key, value)

    entities = []

    add_entity(text, symptom1, "SYMPTOM", entities)

    if symptom2:
        add_entity(text, symptom2, "SYMPTOM", entities)

    add_entity(text, disease, "DISEASE", entities)
    add_entity(text, medication, "MEDICATION", entities)
    add_entity(text, dosage, "DOSAGE", entities)
    add_entity(text, procedure, "PROCEDURE", entities)

    # remove duplicate entities if any
    unique_entities = []
    seen = set()

    for e in entities:
        key = (e["start"], e["end"], e["label"])

        if key not in seen:
            seen.add(key)
            unique_entities.append(e)

    records.append({
        "id": record_id,
        "text": text,
        "entities": unique_entities
    })

In [17]:
# =====================================================
# LOAD AND SAVE JSON
# =====================================================

json_path = RAW_DIR / "synthetic_notes.json"

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(records, file, indent=4)   # Do NOT assign to data


# =====================================================
# SAVE CSV
# =====================================================

csv_records = []

for record in records:   # Use records, not data
    csv_records.append({
        "id": record["id"],
        "text": record["text"]
    })

pd.DataFrame(csv_records).to_csv(
    RAW_DIR / "synthetic_notes.csv",
    index=False
)

print("Dataset generation complete")
print(f"Records: {len(records)}")
print(f"Saved: {json_path}")


Dataset generation complete
Records: 5000
Saved: D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Raw\synthetic_notes.json


### Verify data

In [18]:
# Read json file

json_path = RAW_DIR / "synthetic_notes.json"

with open(json_path, "r", encoding="utf-8") as file:
    data = json.load(file)

In [19]:
len(data)

5000

In [20]:
data[:2]

[{'id': 1,
  'text': 'Known case of Influenza. Advised to take Budesonide 250 mg once daily.',
  'entities': [{'start': 14, 'end': 23, 'label': 'DISEASE'},
   {'start': 41, 'end': 51, 'label': 'MEDICATION'},
   {'start': 52, 'end': 69, 'label': 'DOSAGE'}]},
 {'id': 2,
  'text': 'Clinical impression: prostate cancer. Symptoms include burning urination. Initiate aspirin 1 tablet twice daily.',
  'entities': [{'start': 55, 'end': 72, 'label': 'SYMPTOM'},
   {'start': 21, 'end': 36, 'label': 'DISEASE'},
   {'start': 83, 'end': 90, 'label': 'MEDICATION'},
   {'start': 91, 'end': 111, 'label': 'DOSAGE'}]}]

In [21]:
# ---------- DUPLICATE ERRORS ----------
duplicate_errors = 0
for item in data:
    spans = [
        (e["start"], e["end"], e["label"])
        for e in item.get("entities", [])
    ]
    counts = Counter(spans)
    duplicate_errors += sum(c - 1 for c in counts.values() if c > 1)
    
print(f"Duplicate Errors: {duplicate_errors}")

Duplicate Errors: 0
